# Lab 05 — Structured Streaming, Spark Connect & the Pandas API

**Exam domains:** Structured Streaming (10%) + Spark Connect (5%) + Pandas API on Spark (5%)
= **20% of the exam ≈ 9 questions**

**Time:** ~2 hours

This is your biggest blind spot coming from batch Synapse pipelines. The good news: at Associate level
these questions are mostly conceptual — output modes, watermarks, checkpointing, exactly-once, what
Spark Connect *is*. You do not need to build a production stream. You do need to have run one once so
the vocabulary sticks.

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("lab05-streaming").master("local[4]").getOrCreate()

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
import os, shutil, time

BASE = "/tmp/sparkcert_stream"
shutil.rmtree(BASE, ignore_errors=True)
os.makedirs(f"{BASE}/in", exist_ok=True)
print("scratch:", BASE)

# Part A — Structured Streaming

## A1. The programming model

The one idea to internalise: **a stream is an unbounded table**. Every trigger appends new rows to the
input table; Spark runs your (identical) DataFrame query against it incrementally and updates a result
table; the **output mode** decides what gets written out of that result table.

That's why the API is the same as batch — `select`, `filter`, `groupBy`, `join` all work. The
differences are: `readStream`/`writeStream`, a mandatory **schema** on file sources, **checkpointing**,
**output modes**, **triggers**, and **watermarks**.

**Micro-batch** is the default execution model: Spark plans and runs a small batch job per trigger,
achieving ~100 ms latency. (Continuous processing exists as an experimental at-least-once mode with
~1 ms latency, and is heavily restricted — map-like operations only.)

**Exactly-once end-to-end** requires all three:
1. a **replayable source** (Kafka offsets, file listings — not a socket),
2. a **checkpoint location** (offsets + state, written to durable storage),
3. an **idempotent sink** (Delta, or `foreachBatch` with your own idempotency).

Fault tolerance comes from the checkpoint's **write-ahead log**: on restart Spark reads the last
committed offsets and the state store and resumes exactly there.

In [ ]:
# The simplest possible stream: the built-in `rate` source (n rows per second)
rate = (spark.readStream.format("rate")
        .option("rowsPerSecond", 20)
        .load())          # schema: timestamp TIMESTAMP, value LONG
rate.printSchema()
print("is streaming?", rate.isStreaming)

In [ ]:
counts = (rate
    .withColumn("k", (F.col("value") % 5).cast("int"))
    .groupBy("k").count())

q = (counts.writeStream
     .format("memory")            # writes to an in-memory table you can query with SQL
     .queryName("rate_counts")
     .outputMode("complete")      # aggregation without a watermark => complete
     .option("checkpointLocation", f"{BASE}/ck_rate")
     .trigger(processingTime="2 seconds")
     .start())

time.sleep(10)                                          # let a few micro-batches run
spark.sql("SELECT * FROM rate_counts ORDER BY k").show()
print("status:", q.status)
lp = q.lastProgress          # None until the first micro-batch COMPLETES — always guard this
print("last progress rows/sec:", lp.get("processedRowsPerSecond") if lp else "<no batch yet>")
q.stop()

## A2. Output modes — expect a question

| Mode | Writes | Allowed when |
|---|---|---|
| **append** (default) | only **new** rows, never revised | no aggregation, **or** aggregation **with a watermark** (rows emitted when the window closes) |
| **update** | only rows that **changed** this trigger | any aggregation; the sink must support updates |
| **complete** | the **entire** result table, every trigger | **only** with an aggregation; state grows forever without a watermark |

Traps:
- `complete` on a non-aggregated stream ⇒ `AnalysisException`.
- `append` on an aggregation **without** a watermark ⇒ `AnalysisException` ("Append output mode not
  supported when there are streaming aggregations on streaming DataFrames without watermark").
- `sort`/`orderBy` is only allowed in `complete` mode.
- `count()`, `show()`, `collect()`, `distinct()` and multiple aggregations are not supported on a
  streaming DataFrame — use `writeStream` + `memory`/`console`, or `foreachBatch`.

## A3. Sinks and triggers

**Sinks:** `file` (parquet/json/csv/orc — append only), `kafka`, `delta`, `console` (debug),
`memory` (debug), `foreach` (row-level custom), `foreachBatch` (batch-level custom — this is how you
write to JDBC, do upserts/MERGE, or write to multiple destinations).

**Triggers:**
```python
.trigger(processingTime="30 seconds")   # fixed interval (default is "as fast as possible", 0 seconds)
.trigger(once=True)                     # DEPRECATED — one micro-batch, all available data
.trigger(availableNow=True)             # the modern replacement: process everything, in several
                                        #   micro-batches, then stop. Great for batch-y "incremental" jobs
.trigger(continuous="1 second")         # experimental continuous processing, at-least-once
```
`availableNow` is the one to remember — it gives you streaming's checkpoint bookkeeping with batch's
"run and finish" behaviour, which is exactly the pattern that replaces most Synapse-style incremental
loads.

In [ ]:
# A file source stream — note the MANDATORY schema
schema = StructType([
    StructField("id",    IntegerType()),
    StructField("user",  StringType()),
    StructField("event", StringType()),
    StructField("ts",    TimestampType()),
])

def drop_batch(i, rows):
    (spark.createDataFrame(rows, schema)
          .write.mode("overwrite").json(f"{BASE}/in/batch{i}"))

import datetime as dt
now = dt.datetime(2026, 3, 1, 10, 0, 0)
drop_batch(1, [(1, "ada", "click", now),
               (2, "bram", "view",  now + dt.timedelta(seconds=30)),
               (3, "ada", "click",  now + dt.timedelta(seconds=90))])

stream = (spark.readStream
          .schema(schema)                 # required for file sources; no inference in streaming
          .option("maxFilesPerTrigger", 1)
          .json(f"{BASE}/in"))
print("streaming:", stream.isStreaming)

In [ ]:
# Windowed aggregation with a watermark -> append mode is now legal
windowed = (stream
    .withWatermark("ts", "10 minutes")                     # BEFORE the groupBy, on an event-time column
    .groupBy(F.window("ts", "1 minute", "1 minute"),       # tumbling: slide == window
             F.col("user"))
    .agg(F.count("*").alias("events")))

q2 = (windowed.writeStream
      .format("memory").queryName("win")
      .outputMode("append")
      .option("checkpointLocation", f"{BASE}/ck_win")
      .trigger(availableNow=True)
      .start())
q2.awaitTermination()

spark.sql("SELECT window.start, window.end, user, events FROM win ORDER BY start, user").show(truncate=False)

## A4. Event time, windows and watermarks

```python
F.window(timeColumn, windowDuration, slideDuration=None, startTime=None)
```
- **Tumbling** window: omit `slideDuration` (or set it equal to `windowDuration`) — non-overlapping.
- **Sliding** window: `slideDuration < windowDuration` — a row can land in several windows.
- **Session** window: `F.session_window(ts, "5 minutes")` — gap-based.

The window column is a **struct** with `start` and `end`; select `window.start`, `window.end`.

**Watermark** = `withWatermark("eventTimeCol", "10 minutes")`. It tells Spark: *"I will not wait for data
more than 10 minutes late."* Two consequences:
1. Spark can **drop old state** once `max(event_time_seen) − threshold` passes a window's end —
   this is what bounds state growth.
2. Late rows arriving after that point are **silently dropped**.

`withWatermark` must be called on the same column used in the window, and **before** the aggregation.
Watermark only affects `append`/`update` modes with aggregations; in `complete` mode all state is kept.

**Processing time** = when Spark saw the row. **Event time** = when it happened, a column in the data.
Windows and watermarks always use event time.

In [ ]:
# Streaming deduplication — two variants, both exam-relevant
d1 = stream.dropDuplicates(["id"])                                # unbounded state: every id kept FOREVER
d2 = (stream.withWatermark("ts", "10 minutes")
            .dropDuplicates(["id", "ts"]))                        # state bounded by the watermark
# Spark 3.5 adds:
d3 = (stream.withWatermark("ts", "10 minutes")
            .dropDuplicatesWithinWatermark(["id"]))               # dedup on id only, within the window

for name, d in [("dropDuplicates (no watermark)", d1),
                ("dropDuplicates + watermark", d2),
                ("dropDuplicatesWithinWatermark", d3)]:
    print(f"{name:34} -> {len(d.columns)} cols, streaming={d.isStreaming}")

- `dropDuplicates(cols)` **without** a watermark keeps every key it has ever seen in the state store —
  memory grows without bound. Classic exam answer: "add a watermark".
- `dropDuplicates(["id","ts"])` with a watermark requires the event-time column to be in the key list.
- `dropDuplicatesWithinWatermark(["id"])` (3.5+) does **not** require the timestamp in the key list —
  it dedups on `id` within the watermark delay.

## A5. Stream joins, `foreachBatch`, and monitoring

| Join | Supported? |
|---|---|
| stream ⋈ static, `inner` | ✅ |
| stream ⋈ static, `left` (stream on the left) | ✅ |
| stream ⋈ static, `right`/`full` | ❌ |
| stream ⋈ stream, `inner` | ✅ (watermark recommended to bound state) |
| stream ⋈ stream, `outer` | ✅ but **watermark + time-range condition required** |

In [ ]:
# foreachBatch: the escape hatch. Inside it you have a normal BATCH DataFrame.
def upsert(batch_df, batch_id):
    print(f"batch {batch_id}: {batch_df.count()} rows")     # count() is legal — it's a batch df here
    # batch_df.write.mode("append").jdbc(...)  /  DeltaTable.merge(...)  /  write to 2 sinks

q3 = (stream.writeStream
      .foreachBatch(upsert)
      .option("checkpointLocation", f"{BASE}/ck_fb")
      .trigger(availableNow=True)
      .start())
q3.awaitTermination()

In [ ]:
# Monitoring a query
q4 = (spark.readStream.format("rate").option("rowsPerSecond", 5).load()
      .writeStream.format("memory").queryName("mon")
      .option("checkpointLocation", f"{BASE}/ck_mon").start())
time.sleep(5)

print("id            :", q4.id)          # stable across restarts (tied to the checkpoint)
print("runId         :", q4.runId)       # new on every start
print("isActive      :", q4.isActive)
print("status        :", q4.status)
p = q4.lastProgress
print("inputRowsPerSecond    :", p.get("inputRowsPerSecond"))
print("processedRowsPerSecond:", p.get("processedRowsPerSecond"))
print("batchDuration ms      :", p.get("durationMs"))
print("active queries        :", len(spark.streams.active))
q4.stop()

> **Backpressure signal:** `inputRowsPerSecond` consistently **greater than**
> `processedRowsPerSecond` means you are falling behind. Fix with more resources,
> `maxOffsetsPerTrigger`/`maxFilesPerTrigger` limits, or a simpler query.

> **Checkpoint rule:** one checkpoint location per query, never shared, and it must be on durable
> storage. Changing the query's aggregation or adding/removing stateful operations usually makes the old
> checkpoint incompatible — you must start from a fresh one.

---

# Part B — Spark Connect

## B1. What it is

Spark Connect (GA in Spark 3.4/3.5) decouples the **client** from the **driver**. Your client builds an
**unresolved logical plan**, serialises it as **Protocol Buffers**, and sends it over **gRPC** to a
Spark Connect server that runs inside the driver. Results come back as **Apache Arrow** batches.

Before Connect, your application code *was* the driver — it needed the full Spark JVM in-process, so you
could not embed Spark in an IDE, a notebook server, a web app or a thin container without shipping a
whole Spark installation and colocating with the cluster.

**Benefits (the exam's likely phrasing):**
- **Stability** — a client crash or a memory hog in user code can no longer bring down the driver;
  clients are isolated from each other.
- **Upgradability** — client and server versions are decoupled; the Spark version can move independently.
- **Thin client** — no JVM required on the client (`pip install pyspark-connect`); works from IDEs,
  notebooks, and any language with a gRPC/protobuf implementation.
- **Remote connectivity** — connect to a cluster from anywhere; interactive debugging with breakpoints.

**Limitations:** the low-level **RDD API**, `SparkContext` internals, and accumulators/broadcast via
`sc` are **not available** through Connect. It is a DataFrame/SQL/Structured-Streaming API surface.

```python
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote("sc://myhost:15002").getOrCreate()
# start a server locally:  ./sbin/start-connect-server.sh --packages org.apache.spark:spark-connect_2.12:3.5.0
```

## B2. Deployment modes — know all three

| Mode | Where the **driver** runs | Where **executors** run | Use |
|---|---|---|---|
| **Local** | in your single JVM/process | same JVM as threads (`local[n]`) | dev, tests, small data |
| **Client** | on the **submitting machine** (your laptop / edge node) | on the cluster | interactive: `spark-shell`, notebooks, `pyspark`. Driver logs stream to your terminal; the session dies if your machine disconnects |
| **Cluster** | on a **worker node inside the cluster**, managed by the cluster manager | on the cluster | production `spark-submit`; survives your laptop closing; can be restarted by the cluster manager |

`spark-submit --deploy-mode client|cluster --master yarn|k8s://…|local[*]|spark://…`

Note the two independent axes the exam likes to conflate: `--master` = **which cluster manager**;
`--deploy-mode` = **where the driver lives**. `local[*]` implies client mode.

---

# Part C — Pandas API on Spark

In [ ]:
# NOTE for LOCAL runs: pyspark 3.5's pandas API needs `numpy<2` and `pandas<2.2`
# (`pip install "numpy<2" "pandas<2.2"`). On Databricks the runtime already pins compatible versions.
import pyspark.pandas as ps
import pandas as pd

# Three ways to get a pandas-on-Spark object
psdf1 = ps.DataFrame({"a": [1, 2, 3, 4], "b": ["w", "x", "y", "z"]})
psdf2 = ps.from_pandas(pd.DataFrame({"a": [1, 2], "b": ["p", "q"]}))
sdf   = spark.range(5).withColumn("v", F.col("id") * 2)
psdf3 = sdf.pandas_api()          # Spark DataFrame -> pandas-on-Spark  (was to_pandas_on_spark())

print(type(psdf3))
print(psdf3.head())

In [ ]:
# It looks like pandas, but it runs distributed
print(psdf1.describe())
print(psdf1.groupby("b").sum())
print(psdf1[psdf1.a > 2])
psdf1["c"] = psdf1["a"] * 10
print(psdf1)

# Round-trips
back_to_spark = psdf1.to_spark()          # -> Spark DataFrame
local_pandas  = psdf1.to_pandas()         # -> real pandas: COLLECTS TO THE DRIVER. Dangerous on big data.
print(type(back_to_spark), type(local_pandas))

## C1. Why it exists, and what to remember

**Advantages:** pandas code runs on Spark with (almost) no rewrite; scales past a single machine's
memory; you keep one codebase for small and large data; interoperates freely with Spark DataFrames via
`to_spark()` / `pandas_api()`.

**Differences from real pandas that the exam may probe:**
- **Order is not guaranteed** — Spark is distributed; the default index is not a row order.
- The **index** is emulated (a hidden column), and `compute.ops_on_diff_frames` is **disabled by
  default**, so operations combining two differently-indexed frames raise unless you enable it:
  `ps.set_option("compute.ops_on_diff_frames", True)`.
- `to_pandas()` and `.values` **collect to the driver** — the one call that can kill your job.
- Some pandas APIs (heavily iterative or order-dependent ones) are unimplemented.
- Historically `databricks.koalas`; since Spark 3.2 it is `pyspark.pandas`.

In [ ]:
# Pandas UDFs — the vectorized bridge. Know these three types.
from pyspark.sql.functions import pandas_udf
from typing import Iterator

df = spark.range(1000).withColumn("v", F.rand()).withColumn("g", (F.col("id") % 3).cast("int"))

# 1) Series -> Series  (scalar)
@pandas_udf("double")
def doubled(s: pd.Series) -> pd.Series:
    return s * 2
df.select(doubled("v").alias("d")).show(3)

# 2) Iterator[Series] -> Iterator[Series]  (scalar iterator: init expensive state ONCE per partition)
@pandas_udf("double")
def scaled(it: Iterator[pd.Series]) -> Iterator[pd.Series]:
    factor = 100.0          # imagine loading a model here — done once per partition, not per batch
    for s in it:
        yield s * factor
df.select(scaled("v").alias("s")).show(3)

# 3) Series -> scalar  (aggregate, usable with groupBy and with a Window)
@pandas_udf("double")
def gmean(s: pd.Series) -> float:
    return float(s.mean())
df.groupBy("g").agg(gmean("v").alias("mean_v")).show()

In [ ]:
# 4) applyInPandas: whole group as a pandas DataFrame -> pandas DataFrame ("grouped map")
def normalize(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf["v"] = (pdf["v"] - pdf["v"].mean()) / (pdf["v"].std() or 1.0)
    return pdf

df.groupBy("g").applyInPandas(normalize, schema="id long, v double, g int").show(5)

# 5) mapInPandas: iterate partitions as pandas DataFrames, no grouping
def only_big(it: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    for pdf in it:
        yield pdf[pdf.v > 0.9]
df.mapInPandas(only_big, schema=df.schema).show(5)

| Kind | Signature | Use |
|---|---|---|
| Scalar pandas UDF | `Series → Series` | element-wise column transform |
| Scalar iterator | `Iterator[Series] → Iterator[Series]` | same, but expensive per-partition setup |
| Aggregate pandas UDF | `Series → scalar` | `groupBy().agg()` or `over(window)` |
| `applyInPandas` | `DataFrame → DataFrame`, per group | grouped map; needs an explicit output schema |
| `mapInPandas` | `Iterator[DataFrame] → Iterator[DataFrame]` | per-partition map, same schema shape |

All of them use **Apache Arrow** for zero-copy transfer; batch size is
`spark.sql.execution.arrow.maxRecordsPerBatch` (default 10 000). `applyInPandas` loads **an entire
group** into memory — a skewed group will OOM the Python worker.

### ✍️ Exercises

**5.1** Build a stream over `{BASE}/in` that counts events per 5-minute tumbling window per user, with a
15-minute watermark, in `append` mode, using `trigger(availableNow=True)`. Then drop a new file into the
directory and re-run — verify the checkpoint means only the new file is processed.

**5.2** Explain in one sentence each: why `append` needs a watermark for aggregations; why
`dropDuplicates` without a watermark is dangerous; what `availableNow` gives you that `processingTime`
doesn't.

**5.3** Rewrite `applyInPandas(normalize)` above using a Window function and no Python at all.

<details><summary>Answer to 5.3</summary>

```python
w = Window.partitionBy("g")
df.withColumn("v", (F.col("v") - F.avg("v").over(w)) / F.stddev("v").over(w))
```
Faster: no Arrow serialisation, no Python worker, and Catalyst can optimise it.
</details>

## Wrap-up checklist

- [ ] append / update / complete — and exactly which one an un-watermarked aggregation forbids.
- [ ] The three exactly-once requirements (replayable source, checkpoint, idempotent sink).
- [ ] File streaming sources need an explicit schema.
- [ ] `withWatermark` goes before the aggregation, on the event-time column.
- [ ] `trigger(availableNow=True)` replaces `once=True`.
- [ ] `foreachBatch` gives you a batch DataFrame (so `count()` works there).
- [ ] Spark Connect = gRPC + protobuf plan + Arrow results; no RDD API.
- [ ] Local vs client vs cluster: *where does the driver run*.
- [ ] `pandas_api()` / `to_spark()` / `to_pandas()` — and which one collects to the driver.
- [ ] The five pandas-UDF shapes and what each one's signature looks like.